# 22. Transformer Decoder와 전체 모델

> **제22장** · **이론편 대응: 18장 (Transformer 아키텍처)**
> **예상 소요**: 90분
> **필요 사양**: **[CPU]** 로 실행 가능 (학습 약 2분)
> **추가 설치**: 없음
> **다운로드**: 없음

---

## 이 장에서 하는 일

21장에서 인코더를 완성했다. 디코더는 두 가지가 더 필요하다.

| 절 | 부품 | 왜 필요한가 | 이론편 |
|---|---|---|---|
| 1 | **Masked Self-Attention** | 미래를 미리 보면 안 된다 | 18.5절 |
| 2 | **Encoder-Decoder Attention** | 인코더의 정보를 참조 | 18.2절 |
| 3 | 디코더 블록 조립 | | 18.6절 |
| 4 | 전체 모델 완성 | | 18.2절 |
| 5 | **실제 학습 — 시퀀스 뒤집기** | | |
| 6 | 생성 과정 관찰 | | 18.2절 |
| 7 | PyTorch 내장 구현과 비교 | | |
| 8 | 세 가지 Transformer 유형 | | 19장 예고 |

**1절이 핵심이다.** 디코더가 인코더와 갈리는 지점이 마스킹이며,
이것이 GPT 계열 모델의 작동 방식을 이해하는 열쇠다.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

---

## 1. Masked Self-Attention — 이론편 18.5절 ★

### 왜 가려야 하는가

번역이나 문장 생성은 **한 단어씩 순서대로** 만든다. 세 번째 단어를 만들 때
아직 네 번째 단어는 존재하지 않는다.

그런데 학습할 때는 정답 문장 전체를 갖고 있다. 그대로 넣으면 **모델이 답을 미리 볼 수 있다.**

```
정답:  나는  학교에  간다
       ↑
   이걸 만들 때 '학교에', '간다'를 보면 안 된다
```

이러면 학습 때는 성적이 좋지만, 실제로 생성할 때는 참고할 미래가 없어 무너진다.

**해법**: Attention 점수에서 미래 위치를 $-\infty$로 만든다.
소프트맥스를 거치면 $e^{-\infty} = 0$이 되어 그 위치의 가중치가 0이 된다.

In [ ]:
import torch
import numpy as np


def create_causal_mask(size):
    """미래를 가리는 마스크 (이론편 18.5절)

    아래쪽 삼각형만 True — 자기 자신과 과거만 볼 수 있다.
    """
    return torch.tril(torch.ones(size, size)).bool()


mask = create_causal_mask(5)

print("=" * 55)
print("Causal Mask (미래 차단 마스크)")
print("=" * 55)
print("1 = 볼 수 있음, 0 = 가려짐")
print()
print("       " + "".join(f"{j:>6}" for j in range(5)))
for i in range(5):
    row = "".join(f"{int(v):>6}" for v in mask[i])
    print(f"위치{i}: {row}")
print()
print("읽는 법: 행 = 보는 주체, 열 = 보는 대상")
print("  위치 0은 자기만 볼 수 있다")
print("  위치 4는 0~4 전부 볼 수 있다")
print()
print("대각선 아래쪽만 열려 있어 '하삼각 행렬'이라 부른다.")

In [ ]:
import torch
import numpy as np

print("=" * 60)
print("마스킹이 소프트맥스에 미치는 영향")
print("=" * 60)

torch.manual_seed(0)
scores = torch.randn(5, 5)

print("원래 점수")
print(scores.numpy().round(3))
print()

# 마스크가 0인 곳을 -무한대로
masked_scores = scores.masked_fill(~mask, float("-inf"))
print("마스킹 후 (미래가 -inf)")
print(masked_scores.numpy().round(3))
print()

weights = torch.softmax(masked_scores, dim=-1)
print("소프트맥스 결과")
print(weights.numpy().round(4))
print()

print("각 행이 볼 수 있는 위치 개수")
for i in range(5):
    nonzero = (weights[i] > 1e-9).sum().item()
    print(f"  위치 {i}: {nonzero}개  (합 = {weights[i].sum():.6f})")
print()
print("-" * 60)
assert torch.allclose(weights.sum(-1), torch.ones(5), atol=1e-5)
assert weights[0, 1:].max() < 1e-9, "위치 0이 미래를 보고 있습니다"
print("[OK] 각 행의 합이 1이고, 미래 위치는 정확히 0")
print()
print("-inf 에 exp를 취하면 0이 되므로,")
print("소프트맥스가 자동으로 나머지 위치에만 확률을 배분한다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- 마스크 자체 ---
ax = axes[0]
ax.imshow(mask.numpy(), cmap="Greys_r", vmin=0, vmax=1)
ax.set_title("Causal Mask")
ax.set_xlabel("보는 대상")
ax.set_ylabel("보는 주체")
for i in range(5):
    for j in range(5):
        ax.text(j, i, int(mask[i,j]), ha="center", va="center",
                fontsize=10, color="red" if mask[i,j] else "white")

# --- 마스킹 없는 Attention ---
ax = axes[1]
w_free = torch.softmax(scores, dim=-1).numpy()
im = ax.imshow(w_free, cmap="Blues", vmin=0, vmax=0.6)
ax.set_title("마스킹 없음 (인코더)")
ax.set_xlabel("보는 대상")

# --- 마스킹 있는 Attention ---
ax = axes[2]
im = ax.imshow(weights.numpy(), cmap="Blues", vmin=0, vmax=0.6)
ax.set_title("마스킹 있음 (디코더)")
ax.set_xlabel("보는 대상")
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("오른쪽 그림의 오른쪽 위 삼각형이 완전히 비어 있다.")
print("각 토큰이 자기보다 뒤를 보지 못한다는 뜻이다.")
print()
print("이것이 GPT 계열 모델의 작동 방식이다 (이론편 19.2절).")
print("  '다음 단어 예측'을 학습할 때 미래를 못 보게 해야")
print("  실제 생성 상황과 조건이 같아진다.")

### 왜 `-inf`를 쓰는가

`0`을 넣으면 안 될까? 안 된다. **소프트맥스 전이기 때문**이다.

$e^0 = 1$이므로 점수 0은 "중간 정도 관련 있음"이 되어 버린다.
완전히 차단하려면 $e^x = 0$이 되어야 하고, 그러려면 $x = -\infty$여야 한다.

실무에서는 `-1e9` 같은 아주 작은 수를 쓰기도 한다. 결과는 거의 같다.

In [ ]:
import torch

print("=" * 55)
print("마스크 값에 따른 차이")
print("=" * 55)

s = torch.tensor([2.0, 1.0, 3.0])
print(f"원래 점수: {s.numpy()}")
print("세 번째를 가리고 싶다면")
print()

for name, fill in [("0을 넣으면", 0.0),
                   ("-1e9를 넣으면", -1e9),
                   ("-inf를 넣으면", float("-inf"))]:
    s2 = s.clone()
    s2[2] = fill
    w = torch.softmax(s2, dim=-1)
    print(f"  {name:<16}{w.numpy().round(6)}")

print("-" * 55)
print("0을 넣으면 세 번째가 여전히 9% 정도 확률을 갖는다 — 차단 실패")
print("-1e9 나 -inf 를 넣어야 완전히 0이 된다")

---

## 2. Encoder-Decoder Attention — 이론편 18.2절

디코더에는 Attention이 **두 종류** 들어간다.

| 종류 | Q | K, V | 하는 일 |
|---|---|---|---|
| Masked Self-Attention | 디코더 | **디코더** | 지금까지 만든 것을 참고 |
| **Encoder-Decoder Attention** | 디코더 | **인코더** | 입력 문장을 참고 |

두 번째가 이론편 17장에서 다룬 원래의 Attention이다. 번역으로 치면
"지금 이 단어를 만들 때 원문의 어느 부분을 볼까"를 정하는 것이다.

**Q는 디코더에서, K·V는 인코더에서** 온다는 점이 핵심이다.

In [ ]:
import torch
import torch.nn as nn


class CrossAttention(nn.Module):
    # Encoder-Decoder Attention (이론편 18.2절)
    #
    # Self-Attention과 유일한 차이:
    #   Q는 디코더에서, K와 V는 인코더에서 가져온다.

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split(self, x):
        b, t, _ = x.shape
        return x.view(b, t, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x_decoder, x_encoder, return_weights=False):
        b, t_dec, _ = x_decoder.shape

        Q = self.split(self.W_q(x_decoder))      # 디코더에서
        K = self.split(self.W_k(x_encoder))      # 인코더에서
        V = self.split(self.W_v(x_encoder))      # 인코더에서

        scores = (Q @ K.transpose(-2, -1)) / (self.d_head ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        out = weights @ V

        out = out.transpose(1, 2).contiguous().view(b, t_dec, self.d_model)
        out = self.W_o(out)

        if return_weights:
            return out, weights
        return out


print("=" * 60)
print("Encoder-Decoder Attention")
print("=" * 60)

d_model, n_heads = 64, 4
cross = CrossAttention(d_model, n_heads)

# 입력 문장은 7토큰, 만들고 있는 문장은 5토큰이라고 하자
enc_out = torch.randn(2, 7, d_model)
dec_hidden = torch.randn(2, 5, d_model)

out, w = cross(dec_hidden, enc_out, return_weights=True)

print(f"인코더 출력 : {tuple(enc_out.shape)}   ← 입력 문장 7토큰")
print(f"디코더 상태 : {tuple(dec_hidden.shape)}   ← 만들고 있는 문장 5토큰")
print()
print(f"가중치      : {tuple(w.shape)}  ← (배치, 헤드, 디코더토큰, 인코더토큰)")
print(f"출력        : {tuple(out.shape)}   ← 디코더 모양을 따른다")
print()
print("길이가 달라도 된다는 점에 주목하자.")
print("  Q의 개수가 출력 길이를 정하고, K·V의 개수는 참고할 대상 수다.")
print()
print("여기에는 보통 마스킹을 하지 않는다.")
print("  입력 문장은 이미 전부 주어졌으므로 어디를 봐도 상관없다.")

---

## 3. 디코더 블록 조립 — 이론편 18.6절

인코더 블록에 Cross-Attention을 하나 더 끼워 넣는다.

```
디코더 입력
 ↓
Masked Self-Attention  ← 지금까지 만든 것 참고
 ↓ (+Residual, Norm)
Encoder-Decoder Attention  ← 입력 문장 참고
 ↓ (+Residual, Norm)
Feed-Forward
 ↓ (+Residual, Norm)
출력
```

21장에서 만든 부품을 그대로 쓰되, 하위 층이 세 개가 되었다.

In [ ]:
import torch
import torch.nn as nn


class MaskedSelfAttention(nn.Module):
    # 21장의 MultiHeadAttention에 마스크를 추가한 것

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split(self, x):
        b, t, _ = x.shape
        return x.view(b, t, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x, return_weights=False):
        b, t, _ = x.shape
        Q, K, V = self.split(self.W_q(x)), self.split(self.W_k(x)), self.split(self.W_v(x))

        scores = (Q @ K.transpose(-2, -1)) / (self.d_head ** 0.5)

        # 미래 차단 (1절)
        causal = create_causal_mask(t).to(x.device)
        scores = scores.masked_fill(~causal, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        out = weights @ V
        out = out.transpose(1, 2).contiguous().view(b, t, self.d_model)
        out = self.W_o(out)

        if return_weights:
            return out, weights
        return out


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or d_model * 4
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    # Transformer 디코더 블록 (이론편 18.6절)

    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attn = MaskedSelfAttention(d_model, n_heads)
        self.cross_attn = CrossAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out):
        # 1) 마스크된 Self-Attention
        x = self.norm1(x + self.dropout(self.self_attn(x)))
        # 2) Encoder-Decoder Attention
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out)))
        # 3) Feed-Forward
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x


print("=" * 60)
print("디코더 블록")
print("=" * 60)

block = DecoderBlock(64, 4)
enc_out = torch.randn(2, 7, 64)
dec_in = torch.randn(2, 5, 64)
out = block(dec_in, enc_out)

print(f"디코더 입력 : {tuple(dec_in.shape)}")
print(f"인코더 출력 : {tuple(enc_out.shape)}")
print(f"블록 출력   : {tuple(out.shape)}   ← 디코더 모양 유지")
print()

print("하위 층별 파라미터")
print(f"  Masked Self-Attention   : {sum(p.numel() for p in block.self_attn.parameters()):,}")
print(f"  Encoder-Decoder Attention: {sum(p.numel() for p in block.cross_attn.parameters()):,}")
print(f"  Feed-Forward            : {sum(p.numel() for p in block.ffn.parameters()):,}")
print(f"  LayerNorm x 3           : {sum(p.numel() for p in [*block.norm1.parameters(), *block.norm2.parameters(), *block.norm3.parameters()]):,}")
print(f"  {'합계':<25}: {sum(p.numel() for p in block.parameters()):,}")
print()
print("인코더 블록(21장, 49,984개)보다 Attention 하나만큼 크다.")

---

## 4. 전체 모델 완성 — 이론편 18.2절

인코더와 디코더를 합친다. 이론편 18.2절 그림의 전체 구조다.

```
입력 문장 → [임베딩+PE] → 인코더 블록 xN ─┐
                                          │
정답 문장 → [임베딩+PE] → 디코더 블록 xN ←┘ → 선형 → 확률
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np


def positional_encoding(max_len, d_model):
    # 21장과 같다
    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angles = pos / np.power(10000, (2 * (i // 2)) / d_model)
    PE = np.zeros((max_len, d_model))
    PE[:, 0::2] = np.sin(angles[:, 0::2])
    PE[:, 1::2] = np.cos(angles[:, 1::2])
    return PE


class MultiHeadAttention(nn.Module):
    # 21장의 인코더용 (마스크 없음)
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model, self.n_heads = d_model, n_heads
        self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split(self, x):
        b, t, _ = x.shape
        return x.view(b, t, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x):
        b, t, _ = x.shape
        Q, K, V = self.split(self.W_q(x)), self.split(self.W_k(x)), self.split(self.W_v(x))
        scores = (Q @ K.transpose(-2, -1)) / (self.d_head ** 0.5)
        out = torch.softmax(scores, dim=-1) @ V
        return self.W_o(out.transpose(1, 2).contiguous().view(b, t, self.d_model))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm1(x + self.dropout(self.attn(x)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class Transformer(nn.Module):
    # 전체 Transformer (이론편 18.2절)

    def __init__(self, vocab_size, d_model=64, n_heads=4,
                 n_layers=2, max_len=50, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

        pe = torch.tensor(positional_encoding(max_len, d_model), dtype=torch.float32)
        self.register_buffer("pe", pe)

        self.encoder_blocks = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.decoder_blocks = nn.ModuleList(
            [DecoderBlock(d_model, n_heads, dropout=dropout) for _ in range(n_layers)])

        self.output_layer = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def embed(self, tokens):
        t = tokens.shape[1]
        x = self.embedding(tokens) * (self.d_model ** 0.5)
        x = x + self.pe[:t].unsqueeze(0)
        return self.dropout(x)

    def encode(self, src):
        x = self.embed(src)
        for block in self.encoder_blocks:
            x = block(x)
        return x

    def decode(self, tgt, enc_out):
        x = self.embed(tgt)
        for block in self.decoder_blocks:
            x = block(x, enc_out)
        return self.output_layer(x)

    def forward(self, src, tgt):
        return self.decode(tgt, self.encode(src))


print("=" * 60)
print("전체 Transformer")
print("=" * 60)

VOCAB = 20
model = Transformer(vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=2)

src = torch.randint(0, VOCAB, (2, 7))
tgt = torch.randint(0, VOCAB, (2, 5))
out = model(src, tgt)

print(f"입력 문장 : {tuple(src.shape)}")
print(f"정답 문장 : {tuple(tgt.shape)}")
print(f"출력      : {tuple(out.shape)}  ← (배치, 토큰, 어휘 크기)")
print()
print("출력의 마지막 차원이 어휘 크기인 이유:")
print("  각 위치에서 '다음에 올 단어가 무엇인가'를 어휘 전체에 대한 점수로 낸다.")
print("  소프트맥스를 거치면 확률이 된다 (이론편 20.5절).")
print()
print(f"전체 파라미터: {sum(p.numel() for p in model.parameters()):,}개")

---

## 5. 실제 학습 — 시퀀스 뒤집기

모델이 제대로 동작하는지 확인하려면 실제 과제를 풀려 봐야 한다.

**과제**: 숫자 열을 받아 **거꾸로 출력하기**

```
입력: [8, 3, 6, 6, 10, 6]
출력: [6, 10, 6, 6, 3, 8]
```

이 과제가 좋은 이유가 있다.

- 인코더가 입력 전체를 봐야 한다 (마지막 값이 첫 출력이므로)
- 디코더가 위치를 정확히 알아야 한다
- 정답이 명확해 평가가 쉽다

In [ ]:
import numpy as np
import torch

# 특수 토큰
PAD, SOS, EOS = 0, 1, 2
VOCAB = 12          # 0,1,2는 특수 토큰이므로 실제 숫자는 3~11
SEQ_LEN = 6

def make_reverse_data(n, seed=0):
    rng = np.random.RandomState(seed)
    src = rng.randint(3, VOCAB, size=(n, SEQ_LEN))
    tgt = src[:, ::-1].copy()          # 뒤집기
    return src, tgt


src_train, tgt_train = make_reverse_data(3000, seed=0)
src_test, tgt_test = make_reverse_data(300, seed=1)

print("=" * 60)
print("과제: 시퀀스 뒤집기")
print("=" * 60)
print("예시")
for i in range(3):
    print(f"  {src_train[i]} → {tgt_train[i]}")
print()

# 디코더 입력에는 SOS를 앞에, 정답에는 EOS를 뒤에 붙인다
def prepare(src, tgt):
    n = len(src)
    tgt_input = np.concatenate([np.full((n, 1), SOS), tgt], axis=1)
    tgt_output = np.concatenate([tgt, np.full((n, 1), EOS)], axis=1)
    return (torch.tensor(src), torch.tensor(tgt_input), torch.tensor(tgt_output))


src_t, tgt_in_t, tgt_out_t = prepare(src_train, tgt_train)
src_v, tgt_in_v, tgt_out_v = prepare(src_test, tgt_test)

print("디코더 입출력 구성")
print(f"  입력 문장     : {src_train[0]}")
print(f"  디코더 입력   : {tgt_in_t[0].numpy()}   ← 앞에 SOS(1)")
print(f"  디코더 정답   : {tgt_out_t[0].numpy()}   ← 뒤에 EOS(2)")
print()
print("한 칸씩 밀려 있는 것이 핵심이다 (Teacher Forcing).")
print("  SOS를 보고 → 첫 숫자를 예측")
print("  SOS,첫숫자를 보고 → 둘째 숫자를 예측")
print("  ... 이런 식으로 학습한다.")
print()
print(f"학습 데이터: {len(src_train):,}개 / 시험: {len(src_test):,}개")

### Teacher Forcing

학습할 때 **정답을 디코더 입력으로 준다.** 이를 Teacher Forcing이라 한다.

만약 모델이 만든 것을 다시 입력으로 넣으면, 초반에 한 번 틀리는 순간
그 뒤가 전부 어긋나 학습이 매우 느려진다.

대신 **학습과 실제 생성의 조건이 달라진다**는 문제가 있다.
실제로는 자기가 만든 것을 보고 이어가야 하니까. 이 간극을 노출 편향(exposure bias)이라 부른다.

6절에서 실제 생성을 해 보며 이 차이를 확인한다.

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)
model = Transformer(vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

src_t, tgt_in_t, tgt_out_t = src_t.to(device), tgt_in_t.to(device), tgt_out_t.to(device)
src_v, tgt_in_v, tgt_out_v = src_v.to(device), tgt_in_v.to(device), tgt_out_v.to(device)

EPOCHS = 40
BATCH = 128
history = {"loss": [], "acc": []}

print("=" * 60)
print("학습")
print("=" * 60)
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(src_t))
    total = 0.0
    for i in range(0, len(src_t), BATCH):
        idx = perm[i:i+BATCH]
        optimizer.zero_grad()
        out = model(src_t[idx], tgt_in_t[idx])
        loss = criterion(out.reshape(-1, VOCAB), tgt_out_t[idx].reshape(-1))
        loss.backward()
        optimizer.step()
        total += loss.item() * len(idx)

    # 평가 (Teacher Forcing 상태)
    model.eval()
    with torch.no_grad():
        out_v = model(src_v, tgt_in_v)
        pred = out_v[:, :-1].argmax(-1)         # EOS 예측 부분 제외
        acc = (pred == tgt_out_v[:, :-1]).float().mean().item()

    history["loss"].append(total / len(src_t))
    history["acc"].append(acc)

    if (epoch + 1) % 10 == 0:
        print(f"  에폭 {epoch+1:3}: 손실 {history['loss'][-1]:.4f}  "
              f"토큰 정확도 {acc:.4f}  ({time.time()-t0:.0f}초)")

print("-" * 60)
print(f"최종 토큰 정확도: {history['acc'][-1]:.4f}")
print(f"소요 시간: {time.time()-t0:.0f}초")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["loss"], linewidth=2, color="#1E40AF")
axes[0].set_xlabel("에폭")
axes[0].set_ylabel("손실")
axes[0].set_title("학습 손실")
axes[0].set_yscale("log")
axes[0].grid(alpha=0.3, which="both")

axes[1].plot(history["acc"], linewidth=2, color="#0D9488")
axes[1].axhline(1/9, color="gray", linestyle="--", linewidth=1.5)
axes[1].text(1, 1/9 + 0.03, "무작위 수준", fontsize=8, color="gray")
axes[1].set_xlabel("에폭")
axes[1].set_ylabel("토큰 정확도")
axes[1].set_title("시험 정확도")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. 생성 과정 관찰 — 이론편 18.2절

이제 **실제로 생성**해 본다. Teacher Forcing 없이, 자기가 만든 것을 보고 이어가야 한다.

절차는 이렇다.

1. SOS 하나로 시작
2. 모델에 넣어 다음 토큰을 예측
3. 예측한 것을 뒤에 붙인다
4. EOS가 나오거나 길이 제한까지 2~3 반복

**한 토큰 만들 때마다 모델을 한 번씩 부른다.** 이것이 오늘날 LLM이 답을 한 글자씩
내놓는 이유다(이론편 20.1절).

In [ ]:
import torch


@torch.no_grad()
def generate(model, src, max_len=SEQ_LEN + 1, verbose=False):
    """자기회귀적 생성 (이론편 18.2절, 20.1절)"""
    model.eval()
    enc_out = model.encode(src)                    # 인코더는 한 번만

    b = src.shape[0]
    generated = torch.full((b, 1), SOS, dtype=torch.long, device=src.device)

    steps = []
    for step in range(max_len):
        logits = model.decode(generated, enc_out)  # 지금까지 만든 것 전체를 넣는다
        next_token = logits[:, -1].argmax(-1, keepdim=True)   # 마지막 위치만 사용
        generated = torch.cat([generated, next_token], dim=1)

        if verbose:
            steps.append(generated[0].cpu().numpy().copy())

    return generated[:, 1:], steps      # SOS 제거


# 한 예제의 생성 과정 보기
sample_src = src_v[0:1]
result, steps = generate(model, sample_src, verbose=True)

print("=" * 60)
print("생성 과정 (한 토큰씩)")
print("=" * 60)
print(f"입력 : {sample_src[0].cpu().numpy()}")
print(f"정답 : {tgt_test[0]}")
print()
print("단계별 생성")
for i, s in enumerate(steps):
    shown = " ".join(f"{v:2}" for v in s)
    print(f"  {i+1}단계: [{shown}]")
print()
print(f"최종 : {result[0].cpu().numpy()}")
print()
print("모델을 부른 횟수: 인코더 1번 + 디코더 {}번".format(len(steps)))

In [ ]:
import torch
import numpy as np

# 전체 시험 데이터로 평가
print("=" * 60)
print("실제 생성 성능 (Teacher Forcing 없이)")
print("=" * 60)

generated, _ = generate(model, src_v)
gen_np = generated[:, :SEQ_LEN].cpu().numpy()

token_acc = (gen_np == tgt_test).mean()
exact_match = (gen_np == tgt_test).all(axis=1).mean()

print(f"토큰 단위 정확도 : {token_acc:.4f}")
print(f"완전 일치 비율   : {exact_match:.4f}")
print()
print(f"참고: Teacher Forcing 상태의 정확도는 {history['acc'][-1]:.4f} 였다.")
print()

# 예시 몇 개
print("예시")
print(f"{'입력':<26}{'정답':<26}{'생성'}")
print("-" * 78)
for i in range(6):
    ok = "O" if (gen_np[i] == tgt_test[i]).all() else "X"
    print(f"{str(src_test[i]):<26}{str(tgt_test[i]):<26}{str(gen_np[i])} {ok}")
print("-" * 78)
print()
print("완전 일치는 6개 토큰을 모두 맞혀야 하므로 토큰 정확도보다 낮게 나온다.")
print("토큰 정확도 p 라면 완전 일치는 대략 p^6 이다 (이론편 19.6절 참조).")
print(f"  {token_acc:.4f}^6 = {token_acc**6:.4f}   실제 {exact_match:.4f}")

### Cross-Attention 들여다보기

디코더가 출력을 만들 때 **입력의 어느 부분을 봤는지** 확인한다.
시퀀스 뒤집기 과제이므로, 대각선이 반대 방향으로 나타나야 한다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

model.eval()
sample_src = src_v[0:1]
sample_tgt_in = tgt_in_v[0:1]

with torch.no_grad():
    enc_out = model.encode(sample_src)
    x = model.embed(sample_tgt_in)
    # 첫 블록의 Cross-Attention 가중치
    x = model.decoder_blocks[0].norm1(
        x + model.decoder_blocks[0].self_attn(x))
    _, cross_w = model.decoder_blocks[0].cross_attn(
        x, enc_out, return_weights=True)

attn = cross_w[0].cpu().numpy()      # (헤드, 디코더토큰, 인코더토큰)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for h, ax in enumerate(axes):
    im = ax.imshow(attn[h], cmap="Blues", aspect="auto")
    ax.set_title(f"헤드 {h}", fontsize=10)
    ax.set_xlabel("입력 위치")
    if h == 0:
        ax.set_ylabel("출력 위치")
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN + 1))
    ax.tick_params(labelsize=7)

fig.suptitle("Encoder-Decoder Attention (뒤집기 과제)", fontsize=12)
plt.tight_layout()
plt.show()

print("=" * 55)
print("각 출력 위치가 가장 많이 본 입력 위치")
print("=" * 55)
avg_attn = attn.mean(axis=0)
print(f"{'출력 위치':<12}{'가장 많이 본 입력':<20}{'기대값(뒤집기)'}")
print("-" * 55)
for i in range(SEQ_LEN):
    focus = avg_attn[i].argmax()
    expected = SEQ_LEN - 1 - i
    mark = "  일치" if focus == expected else ""
    print(f"{i:<12}{focus:<20}{expected}{mark}")
print("-" * 55)
print()
print("뒤집기 과제이므로 출력 0번째는 입력 5번째를 봐야 한다.")
print("반대 방향 대각선이 보인다면 모델이 과제 구조를 학습한 것이다.")

---

## 7. PyTorch 내장 구현과 비교

PyTorch에는 `nn.Transformer`가 있다. 직접 만든 것과 파라미터 수를 비교해 보자.

In [ ]:
import torch
import torch.nn as nn

print("=" * 60)
print("직접 구현 vs PyTorch nn.Transformer")
print("=" * 60)

d_model, n_heads, n_layers = 64, 4, 2

builtin = nn.Transformer(
    d_model=d_model, nhead=n_heads,
    num_encoder_layers=n_layers, num_decoder_layers=n_layers,
    dim_feedforward=d_model * 4, batch_first=True)

# 우리 모델에서 임베딩·출력층을 뺀 부분과 비교
ours_core = sum(p.numel() for p in model.encoder_blocks.parameters()) + \
            sum(p.numel() for p in model.decoder_blocks.parameters())
builtin_core = sum(p.numel() for p in builtin.parameters())

print(f"{'':24}{'파라미터'}")
print("-" * 60)
print(f"{'직접 구현 (블록만)':24}{ours_core:,}")
print(f"{'PyTorch nn.Transformer':24}{builtin_core:,}")
print("-" * 60)
print(f"차이: {abs(ours_core - builtin_core):,}")
print()
print("PyTorch 쪽이 조금 더 많은 이유:")
print("  마지막에 LayerNorm을 하나 더 두기 때문이다 (norm 인자로 조절 가능).")
print()

# 사용법 비교
print("=" * 60)
print("사용법")
print("=" * 60)
print("직접 구현")
print("  out = model(src, tgt)")
print()
print("PyTorch 내장 — 마스크를 직접 만들어 넘겨야 한다")
print("  mask = nn.Transformer.generate_square_subsequent_mask(tgt_len)")
print("  out = transformer(src_emb, tgt_emb, tgt_mask=mask)")
print()

# 마스크 형태 확인
m = nn.Transformer.generate_square_subsequent_mask(5)
print("PyTorch가 만드는 마스크 (5x5)")
print(m.numpy())
print()
print("우리가 만든 것과 표현 방식이 다르다.")
print("  우리 : True/False 로 표시하고 masked_fill 로 -inf 채움")
print("  내장 : 처음부터 0 과 -inf 로 만들어 점수에 더함")
print("  결과는 같다.")

---

## 8. 세 가지 Transformer 유형 — 19장 예고

지금까지 인코더와 디코더를 모두 만들었다. 실제 모델들은 **셋 중 하나를 고른다.**

| 유형 | 구성 | 마스킹 | 대표 모델 | 잘하는 일 |
|---|---|---|---|---|
| Encoder만 | 인코더 | 없음 | BERT 계열 | 분류, 이해 |
| **Decoder만** | 디코더 | 있음 | **GPT 계열** | 생성 |
| Encoder-Decoder | 둘 다 | 디코더만 | 번역 모델 | 변환 |

**오늘날 LLM 대부분이 Decoder-only**다. 이론편 19.2절에서 다룬 이유는 이렇다.

- 생성이 목적이므로 마스킹이 필수
- 인코더 없이도 "지금까지의 문맥"을 Self-Attention으로 충분히 볼 수 있다
- 구조가 단순해 크게 키우기 쉽다

Decoder-only 모델을 만들어 보면 차이가 분명해진다.

In [ ]:
import torch
import torch.nn as nn


class DecoderOnlyBlock(nn.Module):
    # GPT 계열의 블록 — Cross-Attention이 없다

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn = MaskedSelfAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Pre-LN 방식 (최근 LLM들이 쓰는 방식, 21장 8절)
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class MiniGPT(nn.Module):
    # 최소한의 GPT 구조 (이론편 19.2절)

    def __init__(self, vocab_size, d_model=64, n_heads=4,
                 n_layers=2, max_len=50):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        pe = torch.tensor(positional_encoding(max_len, d_model), dtype=torch.float32)
        self.register_buffer("pe", pe)
        self.blocks = nn.ModuleList(
            [DecoderOnlyBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens):
        t = tokens.shape[1]
        x = self.embedding(tokens) * (self.d_model ** 0.5)
        x = x + self.pe[:t].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))


print("=" * 60)
print("Decoder-only 모델 (GPT 구조)")
print("=" * 60)

gpt = MiniGPT(vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=2)
tokens = torch.randint(0, VOCAB, (2, 10))
out = gpt(tokens)

print(f"입력: {tuple(tokens.shape)}   ← 문장 하나만 (src/tgt 구분 없음)")
print(f"출력: {tuple(out.shape)}  ← 각 위치에서 다음 토큰 예측")
print()

print("구조 비교")
print(f"{'':24}{'파라미터':<16}{'구성'}")
print("-" * 60)
print(f"{'Encoder-Decoder':24}{sum(p.numel() for p in model.parameters()):<16,}{'인코더 + 디코더'}")
print(f"{'Decoder-only (GPT)':24}{sum(p.numel() for p in gpt.parameters()):<16,}{'디코더만'}")
print("-" * 60)
print()
print("Decoder-only가 훨씬 단순하다.")
print("입력과 출력을 구분하지 않고 하나의 긴 문장으로 다루기 때문이다.")
print()
print("  '번역해줘: 나는 학교에 간다 → I go to school'")
print("  이렇게 이어 붙이면 번역도 '다음 토큰 예측'이 된다.")
print()
print("→ 23장부터 이 구조의 실제 모델들을 다룬다.")

---

## 9. 정리

### 디코더가 인코더와 다른 점

| 항목 | 인코더 (21장) | 디코더 (이 장) |
|---|---|---|
| Self-Attention | 전체를 봄 | **미래를 가림** |
| Cross-Attention | 없음 | **있음** (Q는 디코더, K·V는 인코더) |
| 하위 층 수 | 2개 | 3개 |
| 쓰임 | 이해 | 생성 |

### 확인한 것

| 내용 | 결과 |
|---|---|
| Causal Mask 적용 후 미래 가중치 | 정확히 0 ✓ |
| 마스크 값 (0 vs −inf) | −inf 만 완전 차단 ✓ |
| 시퀀스 뒤집기 학습 | 토큰 정확도 99%대 ✓ |
| Cross-Attention 패턴 | 반대 방향 대각선 ✓ |
| PyTorch 내장과 파라미터 | 거의 일치 (마지막 LayerNorm 차이) ✓ |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 마스킹 | 소프트맥스 전에 −inf, 0은 안 됨 |
| Cross-Attention | **Q는 디코더 / K·V는 인코더** |
| Teacher Forcing | 학습 시 정답을 입력으로 — 빠르지만 노출 편향 |
| 생성 | 토큰 하나마다 모델 호출 (인코더는 한 번) |
| 완전 일치율 | 대략 (토큰 정확도)^길이 |
| Decoder-only | 오늘날 LLM의 주류 구조 |

### Part 4를 마치며

20~22장에서 Transformer를 밑바닥부터 만들었다.

| 장 | 만든 것 |
|---|---|
| 16 | Attention 네 단계 — 이론편 값 검증 |
| 17 | 인코더 — PE·MHA·FFN·Residual·LayerNorm |
| 18 | 디코더 — 마스킹·Cross-Attention·전체 모델 |

**이제 LLM이 어떻게 만들어졌는지 안다.** 다음 부에서는 실제로 학습된
대형 모델을 가져다 쓰면서, 여기서 만든 구조가 그 안에 그대로 있다는 것을 확인한다.

### 다음 장

**23. Hugging Face 기초 — 사전학습 모델 다루기** — 5부가 시작된다.
사전학습된 모델을 내려받아 쓰고, **이론편 20.2절의 BPE 병합 빈도**를 실제 토크나이저로 확인한다.